In [2]:
# train_main.py

import os
import sys
from pathlib import Path

import torch as torch
from torch.utils.data import DataLoader
import torch.nn as nn
from functools import partial
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import numpy as np
from Config import Config
import timm
import torch
import sys
from pathlib import Path

import sys
from pathlib import Path

# Абсолютный путь до папки src
# -------------------------
# Настройка пути до src
# -------------------------
import sys
from pathlib import Path

# Путь до корня проекта (там, где лежит ноутбук)
project_root = Path.cwd()

# Путь до папки src
src_dir = project_root / "src"
sys.path.append(str(src_dir))

# -------------------------
# Импорты
# -------------------------
import torch
from dataset import MultimodalDataset, collate_fn, avg_ingr_cal, get_transforms
from model import MultimodalModel

# Проверка
print("Импорты прошли успешно!")
print("Torch version:", torch.__version__)


# === Тренировка ===
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Загружаем CSV
    # df_all = pd.read_csv(Config.DATA_CSV)
    # DATA_CSV = "c:/Users/Admin/NEIRO/SPR_4_F_2/data/dish.csv"
    df_all = pd.read_csv(Config.DATA_CSV)
    df_train = df_all[df_all["split"]=="train"].reset_index(drop=True)
    df_test  = df_all[df_all["split"]=="test"].reset_index(drop=True)

    # Создаём словарь калорийности для известных ингредиентов
    single = df_train[df_train["ingredients"].str.count(";")==0].copy()
    single["cal_per_g"] = single["total_calories"] / single["total_mass"]
    ingr_cal = {r["ingredients"]: r["cal_per_g"] for _, r in single.iterrows()}

    # Вычисляем avg_ingr_cal и cal_per_100g
    for df in [df_train, df_test]:
        df["avg_ingr_cal"] = df["ingredients"].apply(lambda x: avg_ingr_cal(x, ingr_cal))
        df["cal_per_100g"] = df["total_calories"] / df["total_mass"] * 100


    
    # Трансформации
    train_tfm = get_transforms(Config, ds_type="train")
    test_tfm  = get_transforms(Config, ds_type="test")
    
    # Датасеты
    train_ds = MultimodalDataset(df_train, train_tfm)
    test_ds  = MultimodalDataset(df_test, test_tfm)
    
    # DataLoader
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,
                              collate_fn=partial(collate_fn, tokenizer=train_ds.tokenizer))
    test_loader  = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                              collate_fn=partial(collate_fn, tokenizer=test_ds.tokenizer))
    
    # Модель, оптимизатор, функция потерь
    model = MultimodalModel().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=Config.HEAD_LR)
    loss_fn = nn.L1Loss()
    
    best_val_mae = float("inf")
    
    for e in range(Config.EPOCHS):
        # ===== TRAIN =====
        model.train()
        train_mae = 0
        for b in tqdm(train_loader, desc=f"Epoch {e+1} [Train]"):
            opt.zero_grad()
            preds = model(b["input_ids"].to(device),
                          b["attention_mask"].to(device),
                          b["image"].to(device),
                          b["avg_cal"].to(device),
                          b["group"].to(device))
            labels = b["label"].to(device)
            loss = loss_fn(preds, labels)
            loss.backward()
            opt.step()
            train_mae += torch.mean(torch.abs(preds - labels)).item()
        train_mae /= len(train_loader)
        
        # ===== TEST =====
        model.eval()
        test_mae = 0
        with torch.no_grad():
            for b in test_loader:
                preds = model(b["input_ids"].to(device),
                              b["attention_mask"].to(device),
                              b["image"].to(device),
                              b["avg_cal"].to(device),
                              b["group"].to(device))
                labels = b["label"].to(device)
                test_mae += torch.mean(torch.abs(preds - labels)).item()
        test_mae /= len(test_loader)
        
        print(f"Epoch {e+1} | Train MAE: {train_mae:.2f} | Test MAE: {test_mae:.2f}")
        
train()


Импорты прошли успешно!
Torch version: 2.7.1+cpu


KeyboardInterrupt: 

In [8]:
%reset -f


In [12]:
# train_main.py

import os
import sys
from pathlib import Path

import torch as torch
from torch.utils.data import DataLoader
import torch.nn as nn
from functools import partial
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import numpy as np
from Config import Config
import timm
import torch
import sys
from pathlib import Path

import sys
from pathlib import Path

# Абсолютный путь до папки src
# -------------------------
# Настройка пути до src
# -------------------------
import sys
from pathlib import Path

# Путь до корня проекта (там, где лежит ноутбук)
project_root = Path.cwd()

# Путь до папки src
src_dir = project_root / "src"
sys.path.append(str(src_dir))

# -------------------------
# Импорты
# -------------------------
import importlib
import dataset
importlib.reload(dataset)

from dataset import MultimodalDataset, collate_fn

import torch
from dataset import MultimodalDataset, avg_ingr_cal, get_transforms, collate_fn
print("Collate function:", collate_fn)
from model import MultimodalModel

# Проверка
print("Импорты прошли успешно!")
print("Torch version:", torch.__version__)


# === Тренировка ===
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ===== LOAD DATA =====
    df_all = pd.read_csv(Config.DATA_CSV)
    df_train = df_all[df_all["split"] == "train"].reset_index(drop=True)
    df_test  = df_all[df_all["split"] == "test"].reset_index(drop=True)

    # ===== INGREDIENT CAL DICT =====
    single = df_train[df_train["ingredients"].str.count(";") == 0].copy()
    single["cal_per_g"] = single["total_calories"] / single["total_mass"]
    ingr_cal = {r["ingredients"]: r["cal_per_g"] for _, r in single.iterrows()}

    for df in [df_train, df_test]:
        df["avg_ingr_cal"] = df["ingredients"].apply(lambda x: avg_ingr_cal(x, ingr_cal))
        df["cal_per_100g"] = df["total_calories"] / df["total_mass"] * 100

    # ===== TRANSFORMS =====
    train_tfm = get_transforms(Config, ds_type="train")
    test_tfm  = get_transforms(Config, ds_type="test")

    # ===== DATASETS =====
    train_ds = MultimodalDataset(df_train, train_tfm)
    test_ds  = MultimodalDataset(df_test, test_tfm)

    train_loader = DataLoader(
        train_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        collate_fn=partial(collate_fn, tokenizer=train_ds.tokenizer)
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        collate_fn=partial(collate_fn, tokenizer=test_ds.tokenizer)
    )

    # ===== MODEL =====
    model = MultimodalModel().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=Config.HEAD_LR)
    loss_fn = nn.L1Loss()

    # ===== TRAIN LOOP =====
    for e in range(Config.EPOCHS):

        # ---------- TRAIN ----------
        model.train()
        train_mae_100g = 0.0
        train_mae_dish = 0.0

        batch = next(iter(train_loader))
        # print(f"batch.keys() = {list(batch.keys())}")
# должно вывести: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])


        for b in tqdm(train_loader, desc=f"Epoch {e+1} [Train]"):
            opt.zero_grad()

            preds = model(
                b["input_ids"].to(device),
                b["attention_mask"].to(device),
                b["image"].to(device),
                b["avg_cal"].to(device),
                b["group"].to(device)
            )

            labels = b["label"].to(device)              # kcal / 100g
            mass_100g = b["mass"].to(device) / 100.0   # масса блюда в "100г"

            loss = loss_fn(preds, labels)
            loss.backward()
            opt.step()

            # MAE per 100g
            train_mae_100g += torch.mean(torch.abs(preds - labels)).item()

            # MAE per dish
            dish_error = torch.abs(preds - labels) * mass_100g
            train_mae_dish += dish_error.mean().item()

        train_mae_100g /= len(train_loader)
        train_mae_dish /= len(train_loader)

        # ---------- TEST ----------
        model.eval()
        test_mae_100g = 0.0
        test_mae_dish = 0.0

        with torch.no_grad():
            for b in test_loader:
                preds = model(
                    b["input_ids"].to(device),
                    b["attention_mask"].to(device),
                    b["image"].to(device),
                    b["avg_cal"].to(device),
                    b["group"].to(device)
                )

                labels = b["label"].to(device)
                mass_100g = b["mass"].to(device) / 100.0

                test_mae_100g += torch.mean(torch.abs(preds - labels)).item()
                test_mae_dish += (torch.abs(preds - labels) * mass_100g).mean().item()

        test_mae_100g /= len(test_loader)
        test_mae_dish /= len(test_loader)

        print(
            f"Epoch {e+1} | "
            f"Train MAE: {train_mae_100g:.1f} kcal/100g, {train_mae_dish:.1f} kcal/dish | "
            f"Test MAE: {test_mae_100g:.1f} kcal/100g, {test_mae_dish:.1f} kcal/dish"
        )

train()

Collate function: <function collate_fn at 0x000001DA853DA480>
Импорты прошли успешно!
Torch version: 2.7.1+cpu
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
batch.keys() = ['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass']


Epoch 1 [Train]:   0%|          | 0/173 [00:00<?, ?it/s]

Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])


Epoch 1 [Train]:   1%|          | 1/173 [00:15<43:37, 15.22s/it]

Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])


...

Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])


Epoch 1 [Train]:  99%|█████████▉| 172/173 [23:13<00:08,  8.51s/it]

Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])


Epoch 1 [Train]: 100%|██████████| 173/173 [23:16<00:00,  8.07s/it]


Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys: dict_keys(['input_ids', 'attention_mask', 'image', 'label', 'avg_cal', 'group', 'mass'])
Collate batch keys:

In [ ]:
# train_main_multimodal.py

from pathlib import Path
import sys
# ------------------------- Путь до src -------------------------
project_root = Path.cwd()
src_dir = project_root / "src"
sys.path.append(str(src_dir))

# ------------------------- Импорты -------------------------
import importlib
import torch
import torch.nn as nn
import pandas as pd
import dataset 
import model  # <-- сначала импортируем
importlib.reload(dataset)  # перезагрузка, если нужно
importlib.reload(model)    # теперь можно перезагружать
from dataset import MultimodalDataset, avg_ingr_cal, get_transforms, collate_fn
from torch.utils.data import DataLoader
from functools import partial
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np

import timm

from model import MultimodalModel
import Config          # импортируем модуль целиком
importlib.reload(Config)  # теперь перезагружаем модуль
from Config import Config  # достаем класс после перезагрузки

from dataset import GROUP2ID


from model import MultimodalModel


print("Импорты прошли успешно!")
print("Torch version:", torch.__version__)

# ------------------------- TRAIN FUNCTION -------------------------
def train_model(fusion_type="concat"):
    print(f"\n=== TRAINING MODEL: {fusion_type} ===")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ===== LOAD DATA =====
    df_all = pd.read_csv(Config.DATA_CSV)
    # df_all = pd.read_csv(Config.DATA_CSV, nrows=100)
    df_train = df_all[df_all["split"] == "train"].reset_index(drop=True)
    df_test  = df_all[df_all["split"] == "test"].reset_index(drop=True)

    # ===== INGREDIENT CAL DICT =====
    single = df_train[df_train["ingredients"].str.count(";") == 0].copy()
    single["cal_per_g"] = single["total_calories"] / single["total_mass"]
    ingr_cal = {r["ingredients"]: r["cal_per_g"] for _, r in single.iterrows()}

    for df in [df_train, df_test]:
        df["avg_ingr_cal"] = df["ingredients"].apply(lambda x: avg_ingr_cal(x, ingr_cal))
        df["cal_per_100g"] = df["total_calories"] / df["total_mass"] * 100

    # ===== TRANSFORMS =====
    train_tfm = get_transforms(Config, ds_type="train")
    test_tfm  = get_transforms(Config, ds_type="test")

    # ===== DATASETS & LOADERS =====
    train_ds = MultimodalDataset(df_train, train_tfm)
    test_ds  = MultimodalDataset(df_test, test_tfm)

    train_loader = DataLoader(
        train_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        collate_fn=partial(collate_fn, tokenizer=train_ds.tokenizer)
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        collate_fn=partial(collate_fn, tokenizer=test_ds.tokenizer)
    )

    # ===== MODEL, LOSS, OPTIMIZER =====
    model = MultimodalModel(fusion_type=fusion_type).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=Config.HEAD_LR)
    loss_fn = nn.L1Loss()

    best_mae_100g = float("inf")
    best_epoch_100g = -1

    best_mae_dish = float("inf")
    best_epoch_dish = -1

    # ===== TRAIN LOOP =====
    for e in range(Config.EPOCHS):
        model.train()
        train_mae_100g = 0.0
        train_mae_dish = 0.0

        for b in tqdm(train_loader, desc=f"Epoch {e+1} [Train]"):
            opt.zero_grad()

            preds = model(
                b["input_ids"].to(device),
                b["attention_mask"].to(device),
                b["image"].to(device),
                b["avg_cal"].to(device),
                b["group"].to(device)
            )

            labels = b["label"].to(device)
            mass_100g = b["mass"].to(device) / 100.0

            loss = loss_fn(preds, labels)
            loss.backward()
            opt.step()

            # MAE per 100g
            train_mae_100g += torch.mean(torch.abs(preds - labels)).item()
            # MAE per dish
            train_mae_dish += (torch.abs(preds - labels) * mass_100g).mean().item()

        train_mae_100g /= len(train_loader)
        train_mae_dish /= len(train_loader)

        # ---------- TEST ----------
        model.eval()
        test_mae_100g = 0.0
        test_mae_dish = 0.0

        with torch.no_grad():
            for b in test_loader:
                preds = model(
                    b["input_ids"].to(device),
                    b["attention_mask"].to(device),
                    b["image"].to(device),
                    b["avg_cal"].to(device),
                    b["group"].to(device)
                )

                labels = b["label"].to(device)
                mass_100g = b["mass"].to(device) / 100.0

                test_mae_100g += torch.mean(torch.abs(preds - labels)).item()
                test_mae_dish += (torch.abs(preds - labels) * mass_100g).mean().item()

        test_mae_100g /= len(test_loader)
        test_mae_dish /= len(test_loader)

        if test_mae_100g < best_mae_100g:
            best_mae_100g = test_mae_100g
            best_epoch_100g = e + 1

        if test_mae_dish < best_mae_dish:
            best_mae_dish = test_mae_dish
            best_epoch_dish = e + 1

        print(
            f"Epoch {e+1} | "
            f"Train MAE: {train_mae_100g:.1f} kcal/100g, {train_mae_dish:.1f} kcal/dish | "
            f"Test MAE: {test_mae_100g:.1f} kcal/100g, {test_mae_dish:.1f} kcal/dish"
        )
        
    print(f"\nBest results for fusion='{fusion_type}':")
    print(f"MAE per 100g: {best_mae_100g:.1f} kcal at epoch {best_epoch_100g}")
    print(f"MAE per dish: {best_mae_dish:.1f} kcal at epoch {best_epoch_dish}")

# ------------------------- MAIN -------------------------
if __name__ == "__main__":
    # for fusion in ["concat", "multiply", "cross_attention"]:
    for fusion in ["concat",  "cross_attention"]:
        train_model(fusion_type=fusion)


Импорты прошли успешно!
Torch version: 2.7.1+cpu

=== TRAINING MODEL: concat ===


Epoch 1 [Train]: 100%|██████████| 173/173 [28:19<00:00,  9.82s/it]


Epoch 1 | Train MAE: 54.3 kcal/100g, 93.8 kcal/dish | Test MAE: 46.5 kcal/100g, 68.7 kcal/dish


Epoch 2 [Train]: 100%|██████████| 173/173 [59:45<00:00, 20.72s/it]   


Epoch 2 | Train MAE: 36.9 kcal/100g, 64.9 kcal/dish | Test MAE: 40.1 kcal/100g, 59.1 kcal/dish


...

Epoch 5 [Train]: 100%|██████████| 173/173 [1:10:52<00:00, 24.58s/it]   


Epoch 5 | Train MAE: 23.3 kcal/100g, 44.4 kcal/dish | Test MAE: 35.6 kcal/100g, 55.4 kcal/dish


Epoch 6 [Train]: 100%|██████████| 173/173 [36:51<00:00, 12.78s/it]


Epoch 6 | Train MAE: 21.2 kcal/100g, 40.5 kcal/dish | Test MAE: 30.3 kcal/100g, 53.0 kcal/dish


Epoch 7 [Train]:  19%|█▉        | 33/173 [07:06<29:23, 12.60s/it]

In [ ]:
# train_main_multimodal.py
import importlib
import sys
from pathlib import Path
from functools import partial
from tqdm import tqdm
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np

import dataset
import model
importlib.reload(dataset)
importlib.reload(model)
from dataset import MultimodalDataset, avg_ingr_cal, get_transforms, collate_fn
from model import MultimodalModel
import Config
importlib.reload(Config)
from Config import Config
from dataset import GROUP2ID

print("Импорты прошли успешно!")
print("Torch version:", torch.__version__)

# ------------------------- TRAIN FUNCTION -------------------------
def train_model(fusion_type="concat"):
    print(f"\n=== TRAINING MODEL: {fusion_type} ===")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ===== LOAD DATA =====
    # df_all = pd.read_csv(Config.DATA_CSV)
    df_all = pd.read_csv(Config.DATA_CSV, nrows=10)  # ограничение для теста
    df_train = df_all[df_all["split"] == "train"].reset_index(drop=True)
    df_test  = df_all[df_all["split"] == "test"].reset_index(drop=True)

    # ===== INGREDIENT CAL DICT =====
    single = df_train[df_train["ingredients"].str.count(";") == 0].copy()
    single["cal_per_g"] = single["total_calories"] / single["total_mass"]
    ingr_cal = {r["ingredients"]: r["cal_per_g"] for _, r in single.iterrows()}

    for df in [df_train, df_test]:
        df["avg_ingr_cal"] = df["ingredients"].apply(lambda x: avg_ingr_cal(x, ingr_cal))
        df["cal_per_100g"] = df["total_calories"] / df["total_mass"] * 100

    # ===== TRANSFORMS =====
    train_tfm = get_transforms(Config, ds_type="train")
    test_tfm  = get_transforms(Config, ds_type="test")

    # ===== DATASETS & LOADERS =====
    train_ds = MultimodalDataset(df_train, train_tfm)
    test_ds  = MultimodalDataset(df_test, test_tfm)

    train_loader = DataLoader(
        train_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        collate_fn=partial(collate_fn, tokenizer=train_ds.tokenizer)
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        collate_fn=partial(collate_fn, tokenizer=test_ds.tokenizer)
    )

    # ===== MODEL, LOSS, OPTIMIZER =====
    model = MultimodalModel(fusion_type=fusion_type).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=Config.HEAD_LR)
    loss_fn = nn.L1Loss()

    best_mae_100g = float("inf")
    best_epoch_100g = -1
    best_mae_dish = float("inf")
    best_epoch_dish = -1

    # ===== TRAIN LOOP =====
    for e in range(Config.EPOCHS):
        model.train()
        train_mae_100g = 0.0
        train_mae_dish = 0.0

        for b in tqdm(train_loader, desc=f"Epoch {e+1} [Train]"):
            opt.zero_grad()

            preds = model(
                b["input_ids"].to(device),
                b["attention_mask"].to(device),
                b["image"].to(device),
                b["avg_cal"].to(device),
                b["group"].to(device)
            )

            labels = b["label"].to(device)
            mass_100g = b["mass"].to(device) / 100.0

            loss = loss_fn(preds, labels)
            loss.backward()
            opt.step()

            train_mae_100g += torch.mean(torch.abs(preds - labels)).item()
            train_mae_dish += (torch.abs(preds - labels) * mass_100g).mean().item()

        train_mae_100g /= len(train_loader)
        train_mae_dish /= len(train_loader)

        # ---------- TEST ----------
        model.eval()
        test_mae_100g = 0.0
        test_mae_dish = 0.0

        all_preds = []
        all_labels = []
        all_texts = []
        all_mass = []

        with torch.no_grad():
            for b in test_loader:
                preds = model(
                    b["input_ids"].to(device),
                    b["attention_mask"].to(device),
                    b["image"].to(device),
                    b["avg_cal"].to(device),
                    b["group"].to(device)
                )
                labels = b["label"].to(device)
                mass_100g = b["mass"].to(device) / 100.0

                test_mae_100g += torch.mean(torch.abs(preds - labels)).item()
                test_mae_dish += (torch.abs(preds - labels) * mass_100g).mean().item()

                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())
                all_texts.extend(b["text"])
                all_mass.extend(b["mass"].cpu())

        test_mae_100g /= len(test_loader)
        test_mae_dish /= len(test_loader)

        if test_mae_100g < best_mae_100g:
            best_mae_100g = test_mae_100g
            best_epoch_100g = e + 1

        if test_mae_dish < best_mae_dish:
            best_mae_dish = test_mae_dish
            best_epoch_dish = e + 1

        print(
            f"Epoch {e+1} | "
            f"Train MAE: {train_mae_100g:.1f} kcal/100g, {train_mae_dish:.1f} kcal/dish | "
            f"Test MAE: {test_mae_100g:.1f} kcal/100g, {test_mae_dish:.1f} kcal/dish"
        )

    print(f"\nBest results for fusion='{fusion_type}':")
    print(f"MAE per 100g: {best_mae_100g:.1f} kcal at epoch {best_epoch_100g}")
    print(f"MAE per dish: {best_mae_dish:.1f} kcal at epoch {best_epoch_dish}")

    # ===== ТОП-5 лучших и худших предсказаний с детализацией =====
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_mass = np.array(all_mass)
    errors_100g = np.abs(all_preds - all_labels)
    errors_dish = errors_100g * all_mass / 100.0

    df_results = pd.DataFrame({
        "ingredients": all_texts,
        "pred": all_preds,
        "label": all_labels,
        "mass": all_mass,
        "error_100g": errors_100g,
        "error_dish": errors_dish
    })

    print("\n=== 5 худших предсказаний ===")
    print(df_results.sort_values("error_dish", ascending=False).head(5)[
        ["ingredients", "pred", "label", "mass", "error_100g", "error_dish"]
    ])

    print("\n=== 5 лучших предсказаний ===")
    print(df_results.sort_values("error_dish", ascending=True).head(5)[
        ["ingredients", "pred", "label", "mass", "error_100g", "error_dish"]
    ])


# ------------------------- MAIN -------------------------
if __name__ == "__main__":
    for fusion in ["concat", "multiply", "cross_attention"]:
        train_model(fusion_type=fusion)
